# Lecture: Making CUDA Kernels Fast (Coalescing, Tiling, Tree Reduction)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain the optimization ladder in one line per rung: coalescing feeds the machine (2 vs 32 lines, §4), tiling reuses bytes (§5), trees shorten dependency chains (20 vs 999999 steps at 1M, §6).
- Predict which rung helps a given kernel by naming its starving resource (DRAM transactions, cache reuse, or sequential depth) and its hardware unit (cache lines, shared memory, warps — §4, §5, §7).
- Read the lab's three experiments plus the profiler table (§8: which question smi vs nsys vs ncu answers) and say what each number proves.

## The big idea

Same math can run at wildly different speeds — *how* you touch memory decides. Three tricks form a ladder. **Coalescing** is supermarket packing: memory travels in fixed-size bags (**cache lines**, ~64–128 bytes). If 32 neighboring workers each grab neighboring items, 2 bags suffice; if each grabs one item and skips seven (a **strided** pattern), you haul 32 bags for the same groceries — 16x the traffic for identical floats. **Tiling** is meal-prep: instead of re-reading the whole cookbook for every dish (naive matmul streams each byte O(n) times), stage one reusable chunk in the kitchen's fast shelf (**shared memory** / cache) and cook everything it allows before fetching more — tile size also decides **occupancy**, how many thread-groups stay busy hiding memory waits. **Tree reduction** is a tournament bracket: summing a million numbers one-by-one is 999999 sequential steps (each waits for the last — a **serial dependency chain**), but pairing neighbors halves the field each round, finishing in just 20 rounds (log₂n) with **warp** lanes (32 threads in lockstep, sharing values by shuffle) doing the pairing without touching slow memory. The lab proves each rung with numbers, and **profilers** (`nvidia-smi` for the whole GPU, Nsight Systems/Compute for per-kernel detail) tell you which rung to climb next.

In [ ]:
import numpy as np  # bring in numpy for the cache-line arithmetic
import matplotlib.pyplot as plt  # bring in pyplot for the access-pattern plot
np.random.seed(10)  # fix the seed (pattern below is fixed; seed keeps the habit)
elems = 32  # define floats touched by 32 adjacent threads (one warp-sized group)
line_elems = 16  # define floats per 64-byte cache line (16 fp32 per 64B line)
coal_lines = elems / line_elems  # compute cache lines for contiguous access: 32/16 = 2 lines
stride = 8  # define stride-8 pattern: use 1 float then skip 7 (the bad rung, like the lab)
strided_lines = elems  # compute cache lines for strided access: each float sits alone on its line
print('contiguous lines touched:', coal_lines)  # print 2.0: neighbors share cache lines
print('strided lines touched:', strided_lines)  # print 32: every float wastes a whole line
print('traffic ratio:', round(strided_lines / coal_lines, 1), 'x the cache lines')  # print 16.0x traffic, same floats
print('lab measured strided several-x slower: caches absorb part of the waste')  # connect the sketch to the lab
patterns = ['contiguous good', 'strided-8 bad']  # define bar labels for the two access patterns
lines = [coal_lines, strided_lines]  # collect the cache-line counts per pattern
fig, ax = plt.subplots(figsize=(6, 4))  # create one figure with one axes
ax.bar(patterns, lines, color=['green', 'red'])  # draw the green good bar and the red bad bar
ax.set_ylabel('64-byte cache lines touched for 32 floats')  # label y as cache-line traffic
ax.set_title('Anchor: adjacent threads should touch adjacent bytes (2 vs 32)')  # plain-English title with numbers
ax.grid(True, axis='y', alpha=0.3)  # faint horizontal grid for readability
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 packs the bags, §5 preps the meals, §6 runs the tournament, §7 fields the team, §8 picks the measuring tool.

| Term | One-liner | Section |
|---|---|---|
| coalescing | Adjacent threads touching adjacent bytes — one 128B transaction serves the group | §4 |
| strided / contiguous | Strided: gaps between elements (bad). Contiguous: neighbors in memory (good) | §4 |
| cache line | Fixed-size memory bag (~64B CPU / 128B GPU); you pay per bag, not per byte | §4 |
| tiling / shared memory | Tiling: reuse a staged chunk. Shared memory: fast on-chip shelf per block | §5 |
| occupancy | How many thread-groups stay busy hiding memory waits (tile size decides it) | §5 |
| tree (parallel) reduction | Pairwise tournament summation: log₂n parallel steps instead of n−1 | §6 |
| serial dependency chain | One accumulator where each step waits for the previous — no parallelism | §6 |
| warp | 32 threads executing in lockstep; shuffle shares values without slow memory | §7 |
| nvidia-smi | Driver snapshot: power, temperature, VRAM, utilization (whole GPU) | §8 |
| Nsight (ncu / nsys) | NVIDIA profilers: ncu = per-kernel detail, nsys = whole-timeline trace | §8 |

### §4 Coalescing, stride, and cache lines — packing the bags

> Why 16× in the toy but ~8× in the lab? Same rule, different alignment luck. The toy assumes the worst case — each strided float lands alone on its 64B line (32 lines vs 2 = 16×). The lab's `big[:, ::8]` touches 4-byte floats every 32 bytes, so each 64B line serves *two* touched floats (offsets 0 and 32 inside the line) → 16 lines vs 2 = 8×. Don't memorize the factor — rerun the lab cell and divide *your* two line counts.
**In one sentence: memory moves in fixed-size cache lines (~64 bytes CPU / 128 bytes GPU), so 32 adjacent threads reading 32 adjacent floats touch just 2 lines (coalesced), while the same 32 floats with stride 8 touch 32 lines — 16x the traffic for identical math.**

You pay per bag touched, not per byte used: the strided pattern from the lab (`big[:, ::8]` — one useful 4-byte float per 32-byte span) hauls ~8x the cache lines for the same 524k elements, and the lab measures it several times slower in both NumPy and torch. On GPUs the rule hardens into hardware: one 128-byte transaction serves a whole warp *only if* the warp's addresses are adjacent — otherwise one transaction per thread. Diagnosis: strided is the rung-0 sin; contiguous is rung 1. Watch it: the 2-vs-32 traffic bars.

In [ ]:
import numpy as np  # bring in numpy for the cache-line arithmetic
import matplotlib.pyplot as plt  # bring in pyplot for the traffic bars
np.random.seed(10)  # fix the seed (pattern below is fixed; seed keeps the habit)
elems = 32  # define floats touched by 32 adjacent threads (one warp-sized group)
line_elems = 16  # define floats per 64-byte cache line (16 fp32 per 64B line)
coal_lines = elems / line_elems  # contiguous lines: 32/16 = 2.0 (neighbors share bags)
strided_lines = 32  # strided-8 lines: 32 (each float wastes its own bag)
print('contiguous lines touched:', coal_lines)  # print 2.0: packed bags
print('strided lines touched:', strided_lines)  # print 32: one bag per float
print('traffic ratio:', round(strided_lines / coal_lines, 1), 'x (same 32 floats!)')  # print 16.0x: the packing gap
print('lab fair fight: SAME 524k elements, strided several-x slower (bytes moved lie)')  # print the lab connection
print('gpu rule: one 128B transaction per warp IFF addresses adjacent (coalesced)')  # print the hardware rule
fig, ax = plt.subplots(figsize=(6, 4))  # create one figure with one axes
ax.bar(['contiguous good', 'strided-8 bad'], [coal_lines, strided_lines], color=['green', 'red'])  # green 2 vs red 32
ax.set_ylabel('64-byte cache lines touched for 32 floats')  # label y as cache-line traffic
ax.set_title('Coalescing: adjacent threads should touch adjacent bytes')  # plain-English title
ax.grid(True, axis='y', alpha=0.3)  # faint horizontal grid for readability
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

### §5 Tiling, shared memory, and occupancy — prepping the meals

**In one sentence: tiling restructures loops to reuse a staged chunk (a tile) from fast on-chip shared memory instead of re-streaming it from DRAM, and tile size doubles as the occupancy knob — how many thread-groups stay busy hiding memory waits.**

Naive 96×96 matmul streams each byte O(n) times with zero reuse; tiled code stages a T×T chunk (lab: T=32) in shared memory, cooks every dish it allows, then fetches the next chunk — the same blocking the lab's Python demo runs, where the 96×96 case (216 KB total, fitting in cache) deliberately *ties* because no memory cliff exists for tiling to fix. At 384×384 the tile-structured version runs at ~BLAS speed while raw Python-naive would take ~10 s. GPU moral: `__shared__` tiles are this blocking in kilobytes of SM-local SRAM, and Tensor Cores are the fast tile math. Watch it: the tile diagram showing one C-tile fed by one A-row-strip and one B-column-strip.

In [ ]:
import numpy as np  # reuse numpy for the working-set arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the tile diagram
np.random.seed(11)  # fix the seed (sizes below are fixed; seed keeps the habit)
M, T = 96, 32  # define the lab's small case: 96x96 with 32x32 tiles
kb = (M * M * 8 * 2 + M * M * 8) / 1024  # working set KB: two float64 inputs + one output
print('96x96 float64 working set KB:', round(float(kb), 1))  # print ~216 KB: L2-resident, small cliff
print('tiles per side:', M // T, '-> total tiles:', (M // T) ** 2)  # print 3 -> 9 tiles per matrix
print('naive reuse: each byte streamed O(n) times; tiled: each tile staged once per use')  # print the reuse moral
print('lab result: 96x96 ties (no cliff to fix); 384x384 tiled runs at ~BLAS speed')  # print the honest verdict
fig, ax = plt.subplots(figsize=(6, 5))  # create one square-ish figure for the tile grid
ax.set_xlim(0, 96)  # fix x to matrix columns
ax.set_ylim(0, 96)  # fix y to matrix rows
for k in range(0, 97, 32):  # draw tile grid lines every 32 elements
    ax.plot([k, k], [0, 96], 'k-', lw=1)  # vertical tile edge
    ax.plot([0, 96], [k, k], 'k-', lw=1)  # horizontal tile edge
ax.add_patch(plt.Rectangle((32, 32), 32, 32, facecolor='lightgreen', alpha=0.7))  # highlight one C-tile (the meal)
ax.text(48, 48, 'one C-tile\n32x32', ha='center', va='center', fontsize=9)  # label the staged chunk
ax.text(48, 90, 'A row-strip stages here ->', ha='center', fontsize=8)  # annotate the A feed
ax.text(90, 48, 'B col-strip\nstages here', ha='center', fontsize=8, rotation=90, va='center')  # annotate the B feed
ax.set_xlabel('columns (tiled every 32)')  # label x as tiled columns
ax.set_ylabel('rows (tiled every 32)')  # label y as tiled rows
ax.set_title('Tiling: stage one 32x32 chunk, cook everything, fetch next')  # title the meal-prep story
ax.set_aspect('equal')  # keep tiles square
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram

### §6 Tree vs serial reduction — running the tournament

**In one sentence: a serial sum chains n−1 dependent steps (each waits for the last), while a tree reduction pairs neighbors each round and finishes in log₂n parallel steps — 3 vs 7 on 8 elements, 20 vs 999999 on a million.**

Worked 8: `[3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0, 6.0]` sums to 31.0 either way — optimization never changes answers. Serial: 7 sequential adds. Tree: level 1 makes 4 pairwise sums `[4.0, 5.0, 14.0, 8.0]` in parallel, level 2 makes 2 (`[9.0, 22.0]`), level 3 finishes (31.0). On-GPU each level is one parallel step across blocks, then warp-shuffles, then one value per block to global memory (Project 5's partials pattern). The scaling plot shows the gap exploding: at 1M elements the serial chain needs 999999 steps, the tree 20. Watch it: the worked levels plus the depth plot.

In [ ]:
import numpy as np  # bring in numpy for the step-count math
import matplotlib.pyplot as plt  # bring in pyplot for the depth comparison plot
np.random.seed(11)  # fix the seed (values below are fixed; seed keeps the habit)
vals = np.array([3.0, 1.0, 4.0, 1.0, 5.0, 9.0, 2.0, 6.0])  # define 8 toy numbers to sum
serial = float(vals.sum())  # compute the serial sum in one dependency chain (7 sequential adds)
print('values:', vals.tolist())  # print the 8 numbers being summed
level0 = vals.copy()  # tree level 0: the 8 raw inputs
level1 = level0.reshape(4, 2).sum(axis=1)  # tree level 1: 4 pairwise sums done in parallel
level2 = level1.reshape(2, 2).sum(axis=1)  # tree level 2: 2 pairwise sums done in parallel
level3 = float(level2.sum())  # tree level 3: the final single sum of 2 partials
print('tree levels: 8 ->', level1.tolist(), '->', level2.tolist(), '->', level3)  # print each shrinking level
print('serial total:', serial, '| tree total:', level3)  # print 31.0 vs 31.0: identical answers
print('serial depth: 7 sequential steps vs tree depth: 3 parallel steps')  # print the depth win for 8 elements
ns = np.array([8, 64, 512, 4096, 1024 * 1024])  # define 5 problem sizes for the scaling plot
serial_steps = ns - 1  # serial chain depth: n-1 sequential steps
tree_steps = np.ceil(np.log2(ns)).astype(int)  # tree depth: log2(n) parallel steps
for n_, s, t in zip(ns, serial_steps, tree_steps):  # loop over sizes with both depth counts
    print('n =', int(n_), 'serial steps:', int(s), 'tree steps:', int(t))  # print the growing gap per size
fig, ax = plt.subplots(figsize=(6, 4))  # create one figure with one axes
ax.plot(ns, serial_steps, 'o-', label='serial chain (n-1 steps)')  # draw the linear serial-depth line
ax.plot(ns, tree_steps, 's-', label='tree (log2 n steps)')  # draw the flat tree-depth line
ax.set_xscale('log')  # use log x because sizes span 8 to 1M
ax.set_yscale('log')  # use log y because steps span 3 to 1M
ax.set_xlabel('elements n (log scale)')  # label x as problem size
ax.set_ylabel('sequential steps (log scale, lower is better)')  # label y as dependency-chain depth
ax.set_title('Tree reduction: 1M elements in 20 steps, not 999999')  # title with the headline numbers
ax.legend()  # show the legend for the two summation structures
ax.grid(True, which='both', alpha=0.3)  # faint grid on both log scales
fig.tight_layout()  # fix spacing so labels fit
plt.show()  # display the figure

### §7 Warp — fielding the team of 32

**In one sentence: a warp is 32 threads executing the same instruction in lockstep on NVIDIA GPUs, and warp-shuffle lets those 32 lanes swap values directly through registers — no slow shared or global memory trip.**

Why 32 matters everywhere in this track: one warp-sized group of adjacent threads is exactly the 32-float example in §4 (2 cache lines when coalesced), one row of the hierarchy in Project 5, and one rung of the §6 tree (pairwise adds happen *inside* the warp by shuffle). The lab's 256-thread block is 8 warps; 4096 elements need 128 warps. If threads in a warp take different branches (divergence — e.g. half gulp the `if i < n` guard differently), both paths execute serially and the warp pays double — keep tail code tiny. Watch it: the 32-lane diagram with one shuffle exchange marked.

In [ ]:
import numpy as np  # reuse numpy for the warp-count arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the 32-lane diagram
np.random.seed(12)  # fix the seed (counts below are fixed; seed keeps the habit)
lanes = 32  # define warp width: 32 threads in lockstep (hardware constant)
block = 256  # define lab block size in threads
warps_per_block = block // lanes  # warps per block: 256/32 = 8
n = 4096  # define lab vector length
warps_total = n // lanes  # warps for the whole vector: 4096/32 = 128
print('lanes per warp:', lanes)  # print 32: the lockstep team size
print('warps per 256-block:', warps_per_block)  # print 8: one block fields 8 warps
print('warps for 4096 elements:', warps_total)  # print 128: the full vector's warp count
print('shuffle: lane k reads lane k+16 register (no memory trip)')  # print the shuffle mechanism
fig, ax = plt.subplots(figsize=(9, 2.8))  # make a wide flat figure for the lane diagram
ax.set_xlim(0, 32)  # fix x to the 32 lanes
ax.set_ylim(0, 3)  # fix y for lane boxes + arrow lane
for k in range(32):  # draw one box per lane
    ax.add_patch(plt.Rectangle((k, 1.2), 0.9, 0.9, facecolor='lightblue' if k < 16 else 'wheat'))  # color halves: pairs across halves
    ax.text(k + 0.45, 1.65, str(k), ha='center', va='center', fontsize=7)  # label the lane number
ax.annotate('', xy=(24.5, 2.5), xytext=(8.5, 2.5), arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))  # red shuffle arrow lane 8 <-> 24
ax.text(16, 2.7, 'shuffle: lane 8 <-> lane 24 swap registers (no memory)', ha='center', fontsize=9, color='darkred')  # shuffle label
ax.set_xlabel('warp lanes 0..31 (one instruction, 32 workers)')  # label x as the lockstep team
ax.set_yticks([])  # hide y ticks: only lanes matter
ax.set_title('Warp: 32 lanes in lockstep (256-block = 8 warps, 4096 elems = 128 warps)')  # title with the counts
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram

### §8 Profilers — picking the measuring tool (no code: read the table)

**In one sentence: `nvidia-smi` snapshots the whole GPU (power, temperature, VRAM, utilization), Nsight Systems (`nsys`) traces the whole timeline (kernel overlaps and gaps), and Nsight Compute (`ncu`) dissects one kernel (memory workload, SM throughput, launch stats) — pick the tool that answers your question.**

The lab archives an `smi` snapshot next to the CSV and hands you the DGX runbook for the other two. Read the table row by row: each row is one question you will actually ask, and the check marks say which tool answers it.

| Question you ask | `nvidia-smi` | `nsys` | `ncu` |
|---|---|---|---|
| Is the GPU busy / hot / out of memory right now? | ✅ answers (util %, temp, VRAM used/total, power) | ❌ too fine-grained | ❌ too fine-grained |
| Where do gaps/overlaps in the kernel timeline come from? | ❌ no timeline | ✅ answers (trace CUDA + NVTX, see H2D vs compute) | ❌ single-kernel only |
| Is this kernel DRAM-bound or L1-fed (memory workload)? | ❌ too coarse | ❌ sees traffic, not units | ✅ answers (`memory-workload-analysis`, `l1tex__t_bytes`) |
| Are the SMs fed (throughput) and is the grid big enough? | ❌ too coarse | ⚠️ hints (occupancy over time) | ✅ answers (`sm__throughput`, `launch-stats`) |
| What do I run on DGX Spark? | `nvidia-smi dmon -s pucvmet -d 1` | `nsys profile -o rep --trace=cuda,nvtx …` | `ncu --set detailed -o rep --kernel-name …` |

Do not memorize flags — memorize *which tool for which question*. The §9 ladder tells you what to fix; this table tells you what proves it.

## How it all fits together — the system map §9

One optimization climb is a ladder with every § term as one rung. **Rung 0 → 1** (§4): fix the access pattern — strided (32 lines) to coalesced (2 lines), 16x less traffic for the same floats. **Rung 1 → 2** (§5): add tiling — stage 32×32 chunks in shared memory so each byte is fetched once per use (the 96×96 tie proves ties are data: no cliff, no win). **Rung 2 → 3** (§6): replace the serial chain with a tree — 8 elements in 3 parallel steps not 7, a million in 20 not 999999 — with warp shuffles (§7) doing the pairing. **Measure each rung** (§8): smi says the GPU is fed, nsys says the timeline is tight, ncu says the kernel's memory workload moved. The ladder diagram pins traffic/depth to each rung; the numbers below re-derive the two headline gaps.

In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(9, 4.5))  # tall-ish figure for the four-rung ladder
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 10)  # fix placement range
ax.set_ylim(0, 5)  # fix placement range for 4 rungs
rungs = [('rung 0: naive stride\n32 lines (§4)', 1.0, 'red'), ('rung 1: coalesced\n2 lines, 16x less (§4)', 2.0, 'orange'), ('rung 2: tiled\nfetch once (§5)', 3.0, 'lightgreen'), ('rung 3: tree reduce\n20 not 999999 (§6)', 4.0, 'green')]  # rung label + height + color
for name, y, col in rungs:  # draw each rung as a widening step
    ax.barh(y, y * 2, height=0.6, color=col)  # rung bar: higher rungs reach further right
    ax.text(y * 2 + 0.2, y, name, va='center', fontsize=9)  # rung label with traffic/depth per rung
ax.set_title('System map §9: naive → coalesced → tiled → tree (traffic/depth per rung)')  # diagram title
ax.text(5, 0.3, 'smi → nsys → ncu (§8) prove each rung; warps (§7) climb it', ha='center', fontsize=9, fontweight='bold')  # profiler footnote
plt.tight_layout()  # snug the layout
plt.show()  # render the ladder

In [ ]:
import numpy as np  # reuse numpy for the ladder-gap arithmetic
np.random.seed(9)  # fix the seed (numbers below are fixed; habit kept)
ratio = 32.0 / 2.0  # RUNG 0->1 traffic gap: 32 strided lines vs 2 coalesced lines (§4)
print('coalescing gap:', ratio, 'x less traffic (32 -> 2 lines)')  # print 16.0: the packing win
n = 1_000_000  # ladder scale: one million elements
serial_steps = n - 1  # serial chain depth: 999999 sequential steps (§6)
tree_steps = int(np.ceil(np.log2(n)))  # tree depth: log2(1M) -> 20 parallel steps (§6)
print('serial steps at 1M:', serial_steps, 'vs tree steps:', tree_steps)  # print 999999 vs 20: the depth win
print('tiling gap: 96x96 ties at 216-KB scale (no cliff); 384 wins at ~BLAS speed (§5)')  # print the reuse verdict
print('warp math: 4096 elems / 32 lanes =', 4096 // 32, 'warps do the pairing (§7)')  # print 128: the team count

## Bridge to the lab

Open `notebook.ipynb` now. It climbs the ladder rung by rung, each with a fair CPU-measured number:

1. **Cell 1 (goal markdown)** — the ladder diagram (stride → coalesced → tiled → tree) and the thesis: same FLOPs ≠ same speed (§9).
2. **Cell 2 (harness)** — the two timers: `time_op` (median of repeats for fast kernel-like ops) and `ms_once` (single shot for slow Python loops), both with `synchronize()` (the §7-trio habit from Project 4).
3. **Cell 3 (Experiment A: coalescing)** — equal-element fair fight: a contiguous 524k-element chunk vs a strided 524k-element view (2 vs 32 lines, §4). NumPy strided is several times slower — the bars above, real. GPU rungs included on CUDA boxes.
4. **Cell 4 (Experiment B: tiling)** — naive ijk triple loop vs blocked loop at 96×96 (deliberate tie: cache-resident, so no cliff — the tie *is* the roofline prediction, §5), then tile-structured NumPy vs BLAS at 384×384 (~BLAS speed).
5. **Cell 5 (Experiment C: reduction)** — serial Python loop vs NumPy vs two-level tree vs torch over 4M floats (999999 vs 20 steps, §6) — parallelism structure is performance, paired by warp shuffles (§7). GPU torch-sum row on CUDA boxes.
6. **Cell 6 (CSV + charts)** — `results/opt.csv` and the bandwidth/speedup panels summarizing the whole ladder (§9's traffic/depth per rung).
7. **Cell 7 (profilers)** — `nvidia-smi` snapshot when present, otherwise the DGX runbook (`smi dmon`, `nsys profile`, `ncu --set detailed`) plus what to read: memory-workload, SM-throughput, launch-stats (§8's table tells you which tool).
8. **Cell 8 (README prompts)** — the one-liner: contiguous access feeds the machine (§4), tiling reuses bytes (§5), trees shorten chains (§6); profile to find what starves first (§8).

## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **32 threads read 32 floats with stride 8. How many cache lines move, vs contiguous access — and why does it matter?**

*Answer: 32 lines vs 2 (16x the traffic for identical floats, §4). You pay per cache line touched, so strided access burns DRAM transactions — the red vs green bars above — and one warp needs adjacency for a single 128B transaction.*

2. **Why did the lab's tiled Python matmul tie (not beat) the naive loop at 96×96?**

*Answer: the 96×96 working set is cache-resident, so there is no memory cliff for tiling to fix (§5) — the tie is exactly what the roofline predicts; tiling wins once the working set overflows fast memory (384×384 runs at ~BLAS speed). Tile size still matters on GPU via occupancy and shared-memory staging.*

3. **A serial sum of 1M elements takes 999999 steps. How many does a tree reduction take, which hardware unit pairs values, and which profiler proves the win?**

*Answer: 20 parallel rounds (§6: log₂(1M) ≈ 20 — the depth plot's flat line). Warp-shuffle lanes pair values without touching slow memory (§7: 128 warps cover 4096 elements). Prove it with ncu's memory-workload + SM-throughput for the kernel and nsys for the timeline (§8) — smi only shows the whole-GPU weather.*